# Kirkwood gaps

The overnight sweep in `data/sweep/` puts five runs side by side. Each holds
100 000 test particles in the Sun--Jupiter restricted problem, dumped 64 times
on the way, and each changes exactly one thing:

| directory | e_max | years | perturbers |
|---|---|---|---|
| `e005` | 0.05 | 1 Myr | Jupiter |
| `e010_long` | 0.10 | **3 Myr** | Jupiter |
| `e020` | 0.20 | 1 Myr | Jupiter |
| `e030` | 0.30 | 1 Myr | Jupiter |
| `e010sat` | 0.10 | 1 Myr | Jupiter + Saturn |

The measure throughout is the **depletion ratio**: how many particles sit
within $\pm 0.025$ AU of a resonance at the end, divided by how many sat there
at the start. Four of the five resonances turn out to be a null result, and the
interesting part is how tightly they can be bounded.

All five runs share a seed, so they start from identical particles.

In [ ]:
from matplotlib.colors import LogNorm
import sys

sys.path.insert(0, "python")

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from kirkwood_io import read_epoch, elements, resonance_axes, semimajor_axis

SWEEP = Path("data/sweep")
HALF_WIDTH_AU = 0.025

# Perihelion below this is Mars-crossing; the CPU build culls on
# it. It is a diagonal in the (a, e) plane, which is why that
# plane is worth plotting.
MARS_AU = 1.524

def epoch_at(run, step):
    """The epoch nearest `step`; the runs were dumped on different strides."""
    files = sorted((SWEEP / run).glob("epoch_*.bin"),
                   key=lambda p: int(p.stem.split("_")[1]))
    return read_epoch(min(files, key=lambda p: abs(
        int(p.stem.split("_")[1]) - step)))


first = epoch_at("e005", 0)
RES = resonance_axes(semimajor_axis(first.planet_r[0], first.planet_v[0], first.G))
A0, E0 = elements(first)


def depletion(run, step, label):
    """Counts around one resonance before and after.

    The error is Poisson on the final count, not binomial: the window at the
    end holds particles that drifted in from outside as well as survivors, so
    it is not a fixed set being tracked.
    """
    a, _ = elements(epoch_at(run, step))
    inside = np.abs(a - RES[label]) < HALF_WIDTH_AU
    n1 = int(inside.sum())
    n0 = int((np.abs(A0 - RES[label]) < HALF_WIDTH_AU).sum())
    return n0, n1, n1 / n0, np.sqrt(n1) / n0

## One resonance moves; four do not

At 1 Myr with $e_{\max} = 0.10$, only the 2:1 is visible. The other four sit
within a percent of doing nothing, and the error bars say that is a bound
rather than a coincidence: a window holds about 3300 particles, so anything
deeper than roughly 3.5 % would show at $2\sigma$.

The last two rows change $e_{\max}$ instead, and the 2:1 responds strongly.

In [ ]:
ORDER = ["2:1"] + [k for k in RES if k != "2:1"]

print(f"{'run':>10} {'e_max':>6}   " + "  ".join(f"{k:>13}" for k in ORDER))
for run, e_max in [("e005", "0.05"), ("e010_long", "0.10"),
                   ("e020", "0.20"), ("e030", "0.30")]:
    row = [f"{depletion(run, 36_525_000, k)[2]:6.3f}"
           f"±{depletion(run, 36_525_000, k)[3]:.3f}" for k in ORDER]
    print(f"{run:>10} {e_max:>6}   " + "  ".join(f"{c:>13}" for c in row))

print()
print(f"每个窗口初态约 {int((np.abs(A0 - RES['2:1']) < HALF_WIDTH_AU).sum())} 个粒子，"
      f"所以 1 sigma 约 {np.sqrt(3282) / 3282:.1%}")


## What the ratio looks like

The same five windows, as a histogram of $a$. The 2:1 shows a dip just outside
the resonance line and a pile-up just inside it, which is particles being handed
across the resonance rather than simply lost. As $e_{\max}$ rises the dip fills
in and only the pile-up is left. The other four resonances are flat lines in
every panel.

In [ ]:
TOP = 1500
BINS = np.arange(2.0, 3.51, 0.01)
ORDER = ["2:1", "5:2", "7:3", "3:1", "4:1"]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 6.4), sharex=True, sharey=True)
for ax, (run, e_max) in zip(axes.ravel(),
                            [("e005", "0.05"), ("e010_long", "0.10"),
                             ("e020", "0.20"), ("e030", "0.30")]):
    a_final, _ = elements(epoch_at(run, 36_525_000))
    ax.hist(A0, bins=BINS, color="0.78", label="t = 0")
    ax.hist(a_final, bins=BINS, histtype="step", color="C3", lw=1.3, label="1 Myr")
    for label in ORDER:
        strong = label == "2:1"
        ax.axvline(RES[label], color="C0" if strong else "0.45",
                   lw=1.2 if strong else 0.7, ls=":",
                   alpha=0.9 if strong else 0.7)
        ax.text(RES[label], TOP * 0.99, label, rotation=90, fontsize=7.5,
                color="C0" if strong else "0.35", ha="right", va="top",
                fontweight="bold" if strong else "normal")
    ax.set_ylim(0, TOP)
    ax.set_title(f"$e_{{\\max}}$ = {e_max}", fontsize=10)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(0, 0.87))

for ax in axes[1]:
    ax.set_xlabel("semimajor axis (AU)")
for ax in axes[:, 0]:
    ax.set_ylabel("particles per 0.01 AU")
fig.suptitle("Only the 2:1 opens, and it closes up as the initial eccentricity rises",
             fontsize=11)
fig.tight_layout()
plt.show()

## The same thing in the $(a, e)$ plane

A one-dimensional histogram of $a$ throws away the eccentricity, and the
eccentricity is where the removal criterion lives: Mars crossing is
$a(1-e) < 1.524$, a diagonal in this plane and a single number in the other one.

Two things are visible here that the histogram cannot show. The 2:1 is a wedge
rising in $e$, not a slot at fixed $a$ -- particles are pushed to higher
eccentricity as well as across the resonance. And in the $e_{\max} = 0.30$ panel
there are particles above the Mars line, scattered around $a \approx 2.0$--$2.2$
and near the 2:1, which is the population the removal step later acts on.

In [ ]:
A_EDGES = np.arange(2.0, 3.51, 0.006)
E_EDGES = np.arange(0.0, 0.62, 0.006)

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, (run, e_max) in zip(axes.ravel(),
                            [("e005", "0.05"), ("e010_long", "0.10"),
                             ("e020", "0.20"), ("e030", "0.30")]):
    a_final, e_final = elements(epoch_at(run, 36_525_000))
    ax.hist2d(a_final, e_final, bins=[A_EDGES, E_EDGES],
              norm=LogNorm(vmin=1, vmax=200), cmap="magma")
    for label, axis in RES.items():
        strong = label == "2:1"
        ax.axvline(axis, color="cyan" if strong else "0.7",
                   lw=1.0 if strong else 0.6, ls=":",
                   alpha=0.9 if strong else 0.6)
        ax.text(axis, 0.605, label, rotation=90, fontsize=7,
                color="cyan" if strong else "0.75", ha="right", va="top")
    grid = np.linspace(2.0, 3.5, 200)
    ax.plot(grid, 1 - MARS_AU / grid, color="lime", lw=1.5, label="Mars crossing")
    ax.set_ylim(0, 0.62)
    ax.set_title(f"$e_{{\\max}}$ = {e_max}", fontsize=10)
    ax.legend(fontsize=7, frameon=False, loc="upper right")

for ax in axes[1]:
    ax.set_xlabel("semimajor axis (AU)")
for ax in axes[:, 0]:
    ax.set_ylabel("eccentricity")
fig.suptitle("(a, e) at 1 Myr: the 2:1 is a wedge in e, and the Mars line says "
             "who can be removed", fontsize=11)
fig.tight_layout()
plt.show()

## Is the 3:1 merely slower?

This is the question the earlier 20 000-particle sweep could not answer. If the
3:1 were just slower than the 2:1, tripling the run time should start to show
it. `e010_long` runs three times as long, and nothing appears.

The 2:1, by contrast, reaches its full depth inside the first million years and
then stops moving -- so the timescale on which a resonance of this kind acts is
shorter than 1 Myr, which is exactly why the 3:1's silence at 3 Myr is
informative rather than a matter of patience.

In [ ]:
print(f"{'Myr':>5}  " + "  ".join(f"{k:>14}" for k in RES))
history = {k: [] for k in RES}
for myr in [1, 2, 3]:
    row = []
    for label in RES:
        _, _, ratio, err = depletion("e010_long", int(myr * 36_525_000), label)
        history[label].append(ratio)
        row.append(f"{ratio:6.3f}±{err:.3f}")
    print(f"{myr:>5}  " + "  ".join(f"{c:>14}" for c in row))

fig, ax = plt.subplots(figsize=(6, 3.4))
for label, values in history.items():
    ax.plot([1, 2, 3], values, marker="o", label=label)
ax.axhline(1.0, color="0.7", lw=0.8, zorder=0)
ax.set_xlabel("time (Myr)")
ax.set_ylabel("depletion ratio")
ax.set_xticks([1, 2, 3])
ax.legend(ncol=5, fontsize=8, frameon=False)
ax.set_title("Only the 2:1 moves, and it is done within 1 Myr", fontsize=10)
fig.tight_layout()
plt.show()

## Saturn

Same seed, same 100 000 particles, one extra perturber. The two ratios differ
by less than $1\sigma$, which is the same answer the earlier sweep gave at a
fifth of the statistics -- now with an error bar on it.

In [ ]:
for run in ["e010_long", "e010sat"]:
    perturbers = "Jupiter + Saturn" if run == "e010sat" else "Jupiter"
    _, _, ratio, err = depletion(run, 36_525_000, "2:1")
    print(f"{run:>9}  {perturbers:>16}  2:1 -> {ratio:.3f} ± {err:.3f}")

difference = (depletion("e010sat", 36_525_000, "2:1")[2]
              - depletion("e010_long", 36_525_000, "2:1")[2])
combined = np.hypot(depletion("e010sat", 36_525_000, "2:1")[3],
                    depletion("e010_long", 36_525_000, "2:1")[3])
print(f"\ndifference {difference:+.3f} ± {combined:.3f}"
      f"  ->  {abs(difference) / combined:.1f} sigma")

## Removal can be applied afterwards, and it changes nothing here

The test particles do not interact, so deleting one leaves every other trajectory
untouched. Screening the stored snapshots for Mars crossing is therefore the
same calculation as culling as the run goes, not an approximation of it. A
removal scenario costs seconds against stored trajectories instead of another
night of GPU time. It rests on the restricted-problem assumption and would break
the moment the disk is given self-gravity.

**The step-0 population has to be screened too, and the first version of this
did not.** At $e_{\max} = 0.30$ the inner disk is Mars-crossing from the start:
408 of the 3330 particles in the 4:1 window, 12.3%. Counting those in the
baseline and then counting them again as "removed" manufactures a gap out of the
initial condition -- it reported 0.857, an 8 sigma opening at the 4:1. With the
baseline screened as well the same numbers give **0.977 +/- 0.018**, which is
nothing.

So this section is a null result, and the null result is the useful part: with
the eccentricities this run reaches, no resonance loses a population that could
have stayed.

In [ ]:
def ever_crossing(run):
    """Particles Mars-crossing in any stored snapshot.

    Returns a boolean mask over the particle index. Every run shares a seed, so
    the index ordering is the same across runs.
    """
    ever = None
    for path in sorted((SWEEP / run).glob("epoch_*.bin"),
                       key=lambda f: int(f.stem.split("_")[1])):
        a, e = elements(read_epoch(path))
        hit = a * (1.0 - e) < MARS_AU
        ever = hit if ever is None else (ever | hit)
    return ever


# The step-0 population has to be screened too. At e_max = 0.30 the inner part of
# the disk is Mars-crossing from the start -- 408 of the 3330 particles in the
# 4:1 window, 12% -- and counting those in the baseline and then again as
# "removed" invents a gap out of the initial condition.
SAFE_AT_ZERO = A0 * (1.0 - E0) >= MARS_AU

print(f"{'run':>10} {'e_max':>6} {'crossers':>10} {'of which at t=0':>17}")
masks = {}
for run, e_max in [("e005", "0.05"), ("e010_long", "0.10"),
                   ("e020", "0.20"), ("e030", "0.30")]:
    mask = ever_crossing(run)
    masks[run] = mask
    born = int((mask & ~SAFE_AT_ZERO).sum())
    print(f"{run:>10} {e_max:>6} {mask.sum():>10} {born:>17}")

print()
print("4:1 at e_max = 0.30, before and after removal:")
a_late, e_late = elements(epoch_at("e030", 36_525_000))
mask = masks["e030"]
windows = {}
for label, axis in sorted(RES.items(), key=lambda kv: kv[1]):
    at_zero = (np.abs(A0 - axis) < HALF_WIDTH_AU) & SAFE_AT_ZERO
    counted = np.abs(a_late - axis) < HALF_WIDTH_AU
    kept = counted & ~mask
    windows[label] = (int(at_zero.sum()), int(kept.sum()))
    ratio = kept.sum() / at_zero.sum()
    print(f"  {label:>5}  n0 = {at_zero.sum():>5} (safe at t=0)"
          f"   n1 = {kept.sum():>5} (never crossed)"
          f"   ->  {ratio:.3f} +/- {np.sqrt(kept.sum()) / at_zero.sum():.3f}")
